# 🇧🇩 Bangla Text Generation Fine-tune (Colab Free)

Model: `google/mt5-small` (multilingual, supports Bengali)

**Steps:**
1. Runtime → Change runtime type → T4 GPU
2. Run all cells
3. শেষে নিজের data দিয়ে train করতে পারেন

In [ ]:
!pip install -q transformers datasets accelerate sentencepiece

In [ ]:
from datasets import Dataset

# 🔹 ডেমো ডেটা — আপনি পরে নিজের Bangla text দিয়ে বদলাবেন
texts = [
    "আজকের আবহাওয়া খুব সুন্দর এবং পরিষ্কার আকাশ দেখা যাচ্ছে।",
    "বাংলাদেশ একটি রূপকথার সৌন্দর্যের দেশ যেখানে নদী ও পাহাড় রয়েছে।",
    "শিক্ষা আমাদের জীবনের সবচেয়ে গুরুত্বপূর্ণ অংশ।",
    "ঢাকা বাংলাদেশের রাজধানী এবং সবচেয়ে বড় শহর।",
    "আমি বই পড়তে ভালোবাসি এবং প্রতিদিন নতুন কিছু শেখার চেষ্টা করি।",
    "বাংলাদেশের পহেলা বৈশাখ একটি জাতীয় উৎসব।",
    "স্বাস্থ্যই সকল সুখের মূল, তাই নিয়মিত ব্যায়াম করা উচিত।",
    "বাংলা আমাদের মাতৃভাষা এবং আমরা এতে গর্বিত।",
]

ds = Dataset.from_dict({"text": texts})
ds

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments

model_name = "google/mt5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def tokenize(examples):
    inputs = tokenizer(examples["text"], padding="max_length", truncation=True, max_length=64)
    labels = tokenizer(text_target=examples["text"], padding="max_length", truncation=True, max_length=64)
    inputs["labels"] = labels["input_ids"]
    return inputs

tokenized = ds.map(tokenize, batched=True)
tokenized

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./mt5-bangla",
    per_device_train_batch_size=2,
    num_train_epochs=3,
    logging_steps=5,
    save_steps=50,
    fp16=True,
    predict_with_generate=True,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
)

trainer.train()

In [ ]:
# 🔮 টেস্ট করা
input_text = "বাংলাদেশ"
inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=30)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

## 📝 নিজের ডেটা ব্যবহার করতে:
উপরের `texts` list-এ আপনার Bangla লেখাগুলো দিন। যেমন একটি বই, আর্টিকেল বা গল্পের লাইন।